# **Demo: Structured Prompting and Engineering Tasks**

### **Steps to Follow:**
- **Step 1:** Set up Hugging Face Authentication and Model
- **Step 2:** Define a Function to Get Completion
- **Step 3:** Implement the Four-Element Structured Framework
- **Step 4:** Execute Two Zero-Shot Engineering Tasks
- **Step 5:** Execute a Few-Shot Engineering Task

### **Step 1: Set up Hugging Face Authentication and Model**
- Installs required libraries (`transformers`, `torch`, `accelerate`).
- Securely retrieves the Hugging Face access token from Google Colab Secrets (`HF_TOKEN`).
- Loads `Qwen/Qwen2.5-0.5B-Instruct` onto the GPU with 16-bit half precision.

In [ ]:
# Step 1: Install packages and initialize model pipeline
!pip install -q transformers torch accelerate

import torch
from google.colab import userdata
from transformers import pipeline

# Retrieve token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Initialize text-generation pipeline on GPU
generator = pipeline(
    "text-generation",
    #model="meta-llama/Llama-3.2-3B-Instruct",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    token=hf_token,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.


### **Step 2: Define a Function to Get Completion**
- Wraps the Hugging Face pipeline in an instruction chat format.
- Sets `do_sample=False` to ensure deterministic, reproducible outputs for automated engineering tasks.

In [ ]:
# Step 2: Helper function for chat completions
def get_completion(prompt, max_tokens=150):
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=max_tokens, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

print("✅ Completion helper ready.")

✅ Completion helper ready.


### **Step 3: Implement the Four-Element Structured Framework**
Structured prompts decouple core operational components to ensure clear boundaries:
1. **Instruction:** The primary imperative action.
2. **Context:** System background and technical constraints.
3. **Input Data:** The code snippet, data, or log to process.
4. **Output Indicator:** Format constraints and size boundaries.

In [ ]:
# Step 3: Four-element prompt builder
def build_structured_prompt(instruction, context, input_data, output_indicator):
    return (
        f"Instruction: {instruction}\n"
        f"Context: {context}\n"
        f"Input: {input_data}\n"
        f"Output: {output_indicator}"
    )

print("✅ Structured prompt builder defined.")

✅ Structured prompt builder defined.


### **Step 4: Execute Two Zero-Shot Engineering Tasks**
Zero-shot prompting tests model comprehension relying strictly on explicit instructions and boundaries without demonstration exemplars:
- **Task A (Edge Case Detection):** Spotting potential runtime exceptions (`ZeroDivisionError`).
- **Task B (Security Remediation):** Refactoring string concatenation in SQL queries into parameterized calls (OWASP A03 Injection).

In [ ]:
# --- Task A: Edge Case Detection ---
task_a_prompt = build_structured_prompt(
    instruction="Identify the bug in this function.",
    context="Calculates the average of a list of numbers.",
    input_data="def avg(nums):\n    return sum(nums) / len(nums)",
    output_indicator="Name the breaking edge case in exactly one sentence."
)

print("--- Zero-Shot Task A: Edge Case Detection ---")
print(get_completion(task_a_prompt, max_tokens=35))

# --- Task B: Security Remediation (SQL Injection) ---
task_b_prompt = build_structured_prompt(
    instruction="Fix the SQL injection vulnerability.",
    context="Database lookup query.",
    input_data='query = f"SELECT * FROM users WHERE name = \'{name}\'"',
    output_indicator="Return only the corrected line of code using a parameterized query."
)

print("\n--- Zero-Shot Task B: Security Remediation ---")
print(get_completion(task_b_prompt, max_tokens=150))

--- Zero-Shot Task A: Edge Case Detection ---
The main issue with the provided function is that it does not handle empty lists or lists with non-numeric elements correctly. When an empty list or a list containing only non-

--- Zero-Shot Task B: Security Remediation ---
To fix the SQL injection vulnerability in your database lookup query, you should use a parameterized query instead of concatenating user input directly into the SQL string. Here's how you can do it:

```sql
query = "SELECT * FROM users WHERE name = ?"
```

In this example:
- `?` is a placeholder that will be replaced with the actual value when the query is executed.
- The `?` is used to indicate that the value should be passed as a parameter.

Here’s the corrected version of your original query:

```sql
query = "SELECT * FROM users WHERE name = ?";
```

This approach ensures that any user input (in this case, the variable `name`) is treated as a parameter and not


### **Step 5: Execute a Few-Shot Engineering Task**
Few-shot prompting provides domain-specific demonstrations to establish an exact output format and decision logic:
- **Task C (Automated Defect Triage):** Evaluates incoming feedback, tags the corresponding engineering action (`NONE`, `CREATE_DEFECT`, `UPDATE_DOCS`), and assigns an engineering priority (`P0_BLOCKER` to `P3_LOW`).

In [11]:
# Step 5: Few-shot prompt for engineering defect triage
few_shot_triage_prompt = """
Classify the engineering action and priority based on customer feedback.

Example 1:
Feedback: "Great battery life, works as expected."
Action: NONE | Priority: P3_LOW

Example 2:
Feedback: "The application crashes immediately upon login."
Action: CREATE_DEFECT | Priority: P0_BLOCKER

Example 3:
Feedback: "The user manual has outdated screenshots for setup."
Action: UPDATE_DOCS | Priority: P2_MEDIUM

Current Task:
Feedback: "The item arrived early, but the instructions were missing."
Action:"""

print("--- Few-Shot Task C: Automated Triage ---")
print(get_completion(few_shot_triage_prompt, max_tokens=50))

--- Few-Shot Task C: Automated Triage ---
ACTION: CREATE_DEFECT | PRIORITIES: P0_BLOCKER
